# Error Analysis

Detailed failure analysis for a single model's predictions on the test set.

**Sections:**
1. Setup & load predictions
2. False positive examples (with confidence)
3. False negative examples (missed detections)
4. Analysis by image characteristics (size, brightness)
5. FiftyOne integration (optional)

In [ ]:
import sys
from pathlib import Path

# Add pipeline root to path
PIPELINE_ROOT = Path.cwd().parent if Path.cwd().name == "04_evaluation" else Path.cwd()
sys.path.insert(0, str(PIPELINE_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from utils.config import load_config, resolve_path
from core.p04_evaluation.visualization import draw_bboxes, create_detection_grid

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Configuration

Set the model and dataset to analyze. Update these paths for your specific model.

In [ ]:
# --- Update these for your model ---
MODEL_PATH = "runs/fire/best.pt"          # Path to trained model checkpoint
DATA_CONFIG = PIPELINE_ROOT / "configs" / "data" / "fire.yaml"
SPLIT = "test"
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
MAX_FAILURE_CASES = 100

# Load config
data_config = load_config(DATA_CONFIG)
data_config["_config_dir"] = DATA_CONFIG.parent

class_names = {int(k): v for k, v in data_config["names"].items()}
num_classes = data_config["num_classes"]
print(f"Model: {MODEL_PATH}")
print(f"Dataset: {data_config.get('dataset_name', 'unknown')}")
print(f"Classes: {class_names}")
print(f"Split: {SPLIT}")

## 2. Load Model & Run Inference

Load the trained model and collect predictions + ground truths on the test split.

In [ ]:
import torch
from utils.device import get_device
from evaluate import load_model
from evaluator import ModelEvaluator

device = get_device()
print(f"Device: {device}")

model = load_model(MODEL_PATH, data_config, device)
evaluator = ModelEvaluator(
    model=model,
    data_config=data_config,
    device=device,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
)

# Collect predictions and failure cases
predictions, ground_truths = evaluator.get_predictions(split=SPLIT)
failures = evaluator.get_failure_cases(split=SPLIT, max_cases=MAX_FAILURE_CASES)

n_fp = sum(1 for f in failures if f["type"] == "false_positive")
n_fn = sum(1 for f in failures if f["type"] == "false_negative")
print(f"\nCollected {len(predictions)} images")
print(f"Failure cases: {len(failures)} total ({n_fp} FP, {n_fn} FN)")

## 3. False Positive Analysis

Show images where the model predicted objects that don't exist in the ground truth (or predicted the wrong class).

In [ ]:
fp_cases = [f for f in failures if f["type"] == "false_positive"]

if fp_cases:
    # Confidence distribution of false positives
    fp_scores = [f["score"] for f in fp_cases]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Histogram
    axes[0].hist(fp_scores, bins=20, edgecolor="black", alpha=0.7, color="salmon")
    axes[0].set_xlabel("Confidence Score")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"False Positive Score Distribution (n={len(fp_cases)})")
    axes[0].axvline(x=np.mean(fp_scores), color="red", linestyle="--",
                     label=f"Mean={np.mean(fp_scores):.3f}")
    axes[0].legend()
    
    # Per-class FP breakdown
    fp_by_class = {}
    for f in fp_cases:
        cls_name = class_names.get(f["class_id"], f"class_{f['class_id']}")
        fp_by_class[cls_name] = fp_by_class.get(cls_name, 0) + 1
    
    axes[1].bar(fp_by_class.keys(), fp_by_class.values(), color="salmon", edgecolor="black")
    axes[1].set_ylabel("Count")
    axes[1].set_title("False Positives by Class")
    for i, (name, count) in enumerate(fp_by_class.items()):
        axes[1].text(i, count + 0.5, str(count), ha="center", fontsize=10)
    
    fig.tight_layout()
    plt.show()
    
    # Show top-N highest confidence FPs (most problematic)
    print(f"\nTop 10 highest-confidence false positives:")
    print(f"  {'Rank':>4s} {'Image':>8s} {'Class':<15s} {'Score':>8s}")
    print("  " + "-" * 40)
    fp_sorted = sorted(fp_cases, key=lambda x: x["score"], reverse=True)
    for i, f in enumerate(fp_sorted[:10]):
        cls_name = class_names.get(f["class_id"], f"class_{f['class_id']}")
        print(f"  {i+1:>4d} {f['image_idx']:>8d} {cls_name:<15s} {f['score']:>8.4f}")
else:
    print("No false positives found at the current confidence threshold.")

## 4. False Negative Analysis

Show ground truth objects that the model missed entirely.

In [ ]:
fn_cases = [f for f in failures if f["type"] == "false_negative"]

if fn_cases:
    # Per-class FN breakdown
    fn_by_class = {}
    for f in fn_cases:
        cls_name = class_names.get(f["class_id"], f"class_{f['class_id']}")
        fn_by_class[cls_name] = fn_by_class.get(cls_name, 0) + 1
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(fn_by_class.keys(), fn_by_class.values(), color="steelblue", edgecolor="black")
    ax.set_ylabel("Count")
    ax.set_title(f"False Negatives (Missed Detections) by Class (n={len(fn_cases)})")
    for i, (name, count) in enumerate(fn_by_class.items()):
        ax.text(i, count + 0.5, str(count), ha="center", fontsize=10)
    fig.tight_layout()
    plt.show()
    
    # Box size analysis — are missed objects small?
    fn_areas = []
    for f in fn_cases:
        box = f["box"]
        w = box[2] - box[0]
        h = box[3] - box[1]
        fn_areas.append(w * h)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(fn_areas, bins=30, edgecolor="black", alpha=0.7, color="steelblue")
    ax.set_xlabel("Box Area (pixels^2)")
    ax.set_ylabel("Count")
    ax.set_title("False Negative Box Area Distribution")
    ax.axvline(x=np.median(fn_areas), color="navy", linestyle="--",
               label=f"Median={np.median(fn_areas):.0f}")
    ax.legend()
    fig.tight_layout()
    plt.show()
    
    # Summary table
    print(f"\nFalse Negative Summary:")
    print(f"  {'Class':<15s} {'Count':>8s} {'Avg Area':>12s} {'Min Area':>12s}")
    print("  " + "-" * 50)
    for cls_name in sorted(fn_by_class.keys()):
        cls_fn = [f for f in fn_cases if class_names.get(f["class_id"]) == cls_name]
        areas = [(f["box"][2]-f["box"][0]) * (f["box"][3]-f["box"][1]) for f in cls_fn]
        print(f"  {cls_name:<15s} {len(cls_fn):>8d} {np.mean(areas):>12.0f} {np.min(areas):>12.0f}")
else:
    print("No false negatives found.")

## 5. Analysis by Image Characteristics

Analyze failure rates by image brightness and object size to identify systematic weaknesses.

In [ ]:
# Collect image-level error statistics
# This requires access to the actual images — we use the dataset loader

from utils.metrics import compute_iou

def analyze_image_characteristics(dataloader, predictions, ground_truths, iou_threshold=0.5):
    """Compute per-image brightness and error counts."""
    image_stats = []
    
    for batch_idx, batch in enumerate(dataloader):
        batch_images = batch["images"]  # (B, 3, H, W)
        batch_start = batch_idx * batch_images.shape[0]
        
        for i in range(batch_images.shape[0]):
            img_idx = batch_start + i
            if img_idx >= len(predictions):
                break
            
            # Image brightness (mean pixel value)
            img_np = batch_images[i].numpy().transpose(1, 2, 0)  # (H,W,3)
            brightness = float(np.mean(img_np))
            
            # Count errors for this image
            pred = predictions[img_idx]
            gt = ground_truths[img_idx]
            
            n_pred = pred["boxes"].shape[0]
            n_gt = gt["boxes"].shape[0]
            n_fp = 0
            n_fn = 0
            
            if n_pred > 0 and n_gt > 0:
                iou_mat = compute_iou(pred["boxes"], gt["boxes"])
                gt_matched = np.zeros(n_gt, dtype=bool)
                for pi in range(n_pred):
                    best_gt = -1
                    best_val = iou_threshold
                    for gi in range(n_gt):
                        if gt_matched[gi]:
                            continue
                        if iou_mat[pi, gi] >= best_val:
                            best_val = iou_mat[pi, gi]
                            best_gt = gi
                    if best_gt >= 0:
                        gt_matched[best_gt] = True
                    else:
                        n_fp += 1
                n_fn = int((~gt_matched).sum())
            elif n_pred > 0:
                n_fp = n_pred
            elif n_gt > 0:
                n_fn = n_gt
            
            image_stats.append({
                "idx": img_idx,
                "brightness": brightness,
                "n_pred": n_pred,
                "n_gt": n_gt,
                "n_fp": n_fp,
                "n_fn": n_fn,
            })
    
    return image_stats

# Build dataloader for analysis
try:
    dataloader = evaluator._build_dataloader(SPLIT)
    image_stats = analyze_image_characteristics(
        dataloader, predictions, ground_truths, IOU_THRESHOLD
    )
    
    brightness_vals = [s["brightness"] for s in image_stats]
    fp_counts = [s["n_fp"] for s in image_stats]
    fn_counts = [s["n_fn"] for s in image_stats]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # FP vs brightness
    axes[0].scatter(brightness_vals, fp_counts, alpha=0.4, s=15, color="salmon")
    axes[0].set_xlabel("Mean Brightness")
    axes[0].set_ylabel("False Positives")
    axes[0].set_title("False Positives vs Image Brightness")
    axes[0].grid(True, alpha=0.3)
    
    # FN vs brightness
    axes[1].scatter(brightness_vals, fn_counts, alpha=0.4, s=15, color="steelblue")
    axes[1].set_xlabel("Mean Brightness")
    axes[1].set_ylabel("False Negatives")
    axes[1].set_title("False Negatives vs Image Brightness")
    axes[1].grid(True, alpha=0.3)
    
    fig.tight_layout()
    plt.show()
    
    # Brightness buckets summary
    bright_arr = np.array(brightness_vals)
    fp_arr = np.array(fp_counts)
    fn_arr = np.array(fn_counts)
    
    buckets = [(0, 0.2, "Very Dark"), (0.2, 0.4, "Dark"), (0.4, 0.6, "Medium"),
               (0.6, 0.8, "Bright"), (0.8, 1.0, "Very Bright")]
    
    print(f"\n{'Brightness':<15s} {'Images':>8s} {'Avg FP':>8s} {'Avg FN':>8s}")
    print("-" * 45)
    for lo, hi, name in buckets:
        mask = (bright_arr >= lo) & (bright_arr < hi)
        n = mask.sum()
        if n > 0:
            print(f"{name:<15s} {n:>8d} {fp_arr[mask].mean():>8.2f} {fn_arr[mask].mean():>8.2f}")
        else:
            print(f"{name:<15s} {0:>8d} {'--':>8s} {'--':>8s}")

except Exception as e:
    print(f"Could not load images for characteristic analysis: {e}")
    print("Skipping image-level analysis.")

## 6. FiftyOne Integration (Optional)

Launch FiftyOne for interactive exploration of predictions and failures. This section is optional -- it will fall back gracefully if FiftyOne is not installed.

In [ ]:
try:
    import fiftyone as fo
    
    # Build FiftyOne dataset from predictions
    config_path_str = str(DATA_CONFIG.parent)
    base_path = resolve_path(data_config["path"], config_path_str)
    images_dir = base_path / data_config[SPLIT]
    
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    image_paths = sorted(
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    )
    
    dataset_name = f"{data_config.get('dataset_name', 'model')}_{SPLIT}_analysis"
    
    # Delete existing dataset if it exists
    if dataset_name in fo.list_datasets():
        fo.delete_dataset(dataset_name)
    
    dataset = fo.Dataset(name=dataset_name)
    
    n_samples = min(len(image_paths), len(predictions))
    for i in range(n_samples):
        sample = fo.Sample(filepath=str(image_paths[i]))
        
        pred = predictions[i]
        gt = ground_truths[i]
        
        # Add ground truth
        gt_detections = []
        for j in range(gt["boxes"].shape[0]):
            box = gt["boxes"][j]
            img = cv2.imread(str(image_paths[i]))
            if img is not None:
                h, w = img.shape[:2]
                # Convert xyxy to FiftyOne format [x, y, width, height] (relative)
                rel_box = [
                    float(box[0]) / w,
                    float(box[1]) / h,
                    float(box[2] - box[0]) / w,
                    float(box[3] - box[1]) / h,
                ]
            else:
                rel_box = [0, 0, 0, 0]
            
            label = class_names.get(int(gt["labels"][j]), str(gt["labels"][j]))
            gt_detections.append(fo.Detection(label=label, bounding_box=rel_box))
        
        sample["ground_truth"] = fo.Detections(detections=gt_detections)
        
        # Add predictions
        pred_detections = []
        for j in range(pred["boxes"].shape[0]):
            box = pred["boxes"][j]
            img = cv2.imread(str(image_paths[i]))
            if img is not None:
                h, w = img.shape[:2]
                rel_box = [
                    float(box[0]) / w,
                    float(box[1]) / h,
                    float(box[2] - box[0]) / w,
                    float(box[3] - box[1]) / h,
                ]
            else:
                rel_box = [0, 0, 0, 0]
            
            label = class_names.get(int(pred["labels"][j]), str(pred["labels"][j]))
            pred_detections.append(fo.Detection(
                label=label,
                bounding_box=rel_box,
                confidence=float(pred["scores"][j]),
            ))
        
        sample["predictions"] = fo.Detections(detections=pred_detections)
        dataset.add_sample(sample)
    
    # Evaluate with FiftyOne built-in metrics
    fo_results = dataset.evaluate_detections(
        "predictions",
        gt_field="ground_truth",
        eval_key="eval",
        iou=IOU_THRESHOLD,
    )
    
    print(f"FiftyOne dataset created: '{dataset_name}' ({len(dataset)} samples)")
    print(f"Launch the FiftyOne app with: fo.launch_app(dataset)")
    print(f"Or in the terminal: fiftyone app launch")
    
    # Show FP/FN counts from FiftyOne evaluation
    fo_results.print_report()

except ImportError:
    print("FiftyOne is not installed. Install with: pip install fiftyone")
    print("Skipping FiftyOne integration. The analysis above provides")
    print("equivalent failure case information via matplotlib.")

except Exception as e:
    print(f"FiftyOne error: {e}")
    print("Skipping FiftyOne integration.")